# 🦜 RAG with LangChain

**Day 7 — Notebook 3 of 4 | Estimated Time: 35 minutes**

---

## What You'll Learn
- Why LangChain exists and what it adds over raw API calls
- LangChain document loaders and text splitters
- Building a retriever with LangChain + ChromaDB
- The LCEL (LangChain Expression Language) RAG chain pattern
- Streaming responses in a RAG pipeline

---

## Setup

In [ ]:
%pip install -U -q langchain langchain-google-genai langchain-chroma langchain-community chromadb

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from IPython.display import Markdown

API_KEY = get_api_key()
os.environ["GOOGLE_API_KEY"] = API_KEY
print("✅ Ready!")

---

## 1. Why LangChain?

In Notebook 2 we built RAG from scratch. LangChain provides pre-built components that:
- Handle the boilerplate (embedding, chunking, retrieval)
- Provide a composable chain abstraction (LCEL)
- Offer 100+ integrations (vector DBs, LLMs, document loaders)
- Enable streaming, callbacks, tracing, and observability

```
From scratch:          LangChain equivalent:
─────────────────      ─────────────────────────────────────
embed_texts()      →   GoogleGenerativeAIEmbeddings
chromadb.Client()  →   Chroma vector store
RecursiveChunk()   →   RecursiveCharacterTextSplitter
manual loop        →   retriever | prompt | llm | parser  (LCEL)
```

---

## 2. LangChain Document Loaders

LangChain has loaders for many formats. We'll use the basic `Document` class and `TextLoader`.

In [ ]:
import pathlib
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create sample text files to load
DOCS_DIR = pathlib.Path("/tmp/lc_rag_docs")
DOCS_DIR.mkdir(exist_ok=True)

sample_docs = {
    "climate_basics.txt": """Climate Change Basics

Climate change refers to long-term shifts in global temperatures and weather patterns. While some 
climate change is natural, since the 1800s human activities have been the main driver, primarily 
through the burning of fossil fuels like coal, oil, and gas.

Burning fossil fuels generates greenhouse gas emissions that act like a blanket wrapped around 
the Earth, trapping the sun's heat and raising temperatures. The main greenhouse gases are 
carbon dioxide (CO2) and methane. CO2 is released through burning of fossil fuels, deforestation, 
and land use changes. Methane is emitted mainly from livestock farming, landfills, and the 
fossil fuel industry.

The effects of climate change include more frequent and intense weather events, rising sea levels, 
melting ice caps, and disruptions to ecosystems. The Intergovernmental Panel on Climate Change 
(IPCC) warns that limiting warming to 1.5°C requires rapid, far-reaching changes across all 
sectors of society.""",

    "renewable_energy.txt": """Renewable Energy Sources

Renewable energy comes from sources that are naturally replenished: sunlight, wind, rain, tides, 
waves, and geothermal heat. Unlike fossil fuels, renewable energy sources are sustainable and 
produce little to no greenhouse gas emissions during operation.

Solar power uses photovoltaic cells or concentrated solar thermal systems to convert sunlight 
into electricity. Solar capacity has grown exponentially, with costs falling over 90% in the 
last decade. Wind power harnesses kinetic energy from moving air using turbines. Offshore wind 
farms can generate more consistent power than onshore installations due to stronger, steadier winds.

Hydropower is the largest source of renewable electricity globally, using water flow to drive 
turbines. Geothermal energy taps heat from within the Earth, providing base-load power in 
volcanic regions. Battery storage technologies are increasingly paired with renewables to address 
intermittency challenges.""",

    "carbon_capture.txt": """Carbon Capture and Storage

Carbon capture and storage (CCS) involves capturing CO2 emissions from industrial sources and 
storing them underground to prevent them from entering the atmosphere. The technology has three 
stages: capture, transport, and storage.

Post-combustion capture separates CO2 from flue gases after fossil fuels have been burned. 
Pre-combustion capture converts fuel into hydrogen and CO2 before combustion, capturing the CO2 
and burning clean hydrogen. Oxyfuel combustion burns fuel in pure oxygen, producing a concentrated 
CO2 stream for easier capture.

Direct air capture (DAC) removes CO2 directly from the atmosphere, allowing negative emissions. 
While DAC technology is proven, current costs are high — around $300-600 per tonne of CO2. 
Scaling CCS is seen as essential for meeting net-zero targets, particularly for hard-to-abate 
sectors like steel, cement, and aviation.""",
}

for fname, content in sample_docs.items():
    (DOCS_DIR / fname).write_text(content)

# Load with TextLoader
all_lc_docs = []
for fpath in sorted(DOCS_DIR.glob("*.txt")):
    loader = TextLoader(str(fpath))
    docs = loader.load()
    # Add source metadata
    for doc in docs:
        doc.metadata["source"] = fpath.name
    all_lc_docs.extend(docs)

print(f"Loaded {len(all_lc_docs)} documents")
for doc in all_lc_docs:
    print(f"  {doc.metadata['source']}: {len(doc.page_content)} chars")

---

## 3. Splitting and Embedding with LangChain

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks = splitter.split_documents(all_lc_docs)
print(f"Split into {len(chunks)} chunks")

# LangChain embedding wrapper for Gemini
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004",
    task_type="retrieval_document",
    google_api_key=API_KEY,
)

# Create ChromaDB vector store via LangChain
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="climate_docs",
)

print(f"✅ Vector store created with {vectorstore._collection.count()} chunks")

---

## 4. The LangChain Retriever

In [ ]:
# Create a retriever from the vector store
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

# Test retrieval
query = "How does solar power work and why have costs fallen?"
retrieved_docs = retriever.invoke(query)

print(f"Query: '{query}'")
print(f"Retrieved {len(retrieved_docs)} chunks:\n")
for i, doc in enumerate(retrieved_docs):
    print(f"  Chunk {i+1} [source: {doc.metadata.get('source', 'unknown')}]:")
    print(f"  {doc.page_content[:120]}...")
    print()

---

## 5. Building the RAG Chain with LCEL

**LCEL (LangChain Expression Language)** lets you compose chains using `|` (pipe) operators — similar to Unix pipes.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. The LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,
    google_api_key=API_KEY,
)

# 2. The prompt template
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a knowledgeable assistant on climate science and energy.
Answer using ONLY the provided context. Be concise and cite the source document.
If the answer is not in the context, say: "I don't have information on that in the provided documents."""),
    ("human", """Context:
{context}

Question: {question}""")
])


def format_docs(docs: list) -> str:
    """Format retrieved documents into a single context string."""
    return "\n\n---\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )


# 3. Compose the RAG chain with LCEL
rag_chain = (
    {
        "context": retriever | format_docs,   # Retrieve + format
        "question": RunnablePassthrough(),    # Pass question through unchanged
    }
    | RAG_PROMPT                             # Build the prompt
    | llm                                    # Call Gemini
    | StrOutputParser()                      # Extract the string
)

print("RAG chain built. Components:")
print("  retriever → format_docs → RAG_PROMPT → llm → StrOutputParser")

In [ ]:
# Run the chain
questions = [
    "What are the main greenhouse gases and where do they come from?",
    "Why is offshore wind better than onshore wind?",
    "What is direct air capture and how much does it cost?",
    "How is hydropower different from solar power?",
]

for q in questions:
    answer = rag_chain.invoke(q)
    print(f"❓ {q}")
    print(f"🤖 {answer.strip()}")
    print("-" * 60)

---

## 6. Returning Sources Alongside Answers

In production you often want to return not just the answer, but also which sources were used.

In [ ]:
from langchain_core.runnables import RunnableParallel

# Chain that returns both the answer AND the source documents
rag_chain_with_sources = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(
    answer=(
        lambda x: RAG_PROMPT.invoke({
            "context": format_docs(x["context"]),
            "question": x["question"],
        })
        | llm
        | StrOutputParser()
    )
)

result = rag_chain_with_sources.invoke(
    "What needs to happen to limit warming to 1.5 degrees?"
)

print("Answer:")
print(result["answer"])
print("\nSources used:")
for doc in result["context"]:
    print(f"  - {doc.metadata.get('source')} (chunk: {doc.page_content[:60]}...)")

---

## 7. Streaming Responses

In [ ]:
import sys

print("Streaming answer:\n")

# Stream tokens as they arrive
for chunk in rag_chain.stream("Explain the three stages of carbon capture and storage."):
    print(chunk, end="", flush=True)

print("\n\n✅ Streaming complete")

---

## 8. Metadata Filtering with LangChain + ChromaDB

In [ ]:
# Filter retrieval to a specific source file
filtered_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3,
        "filter": {"source": "renewable_energy.txt"},  # Only search this file
    },
)

# Build a chain with filtered retriever
filtered_chain = (
    {"context": filtered_retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("Querying ONLY renewable_energy.txt:\n")
answer = filtered_chain.invoke("What are the challenges with renewable energy?")
print(answer)

---

## 🏋️ Exercise 1: Build a RAG Chain for a Different Domain

Apply the LangChain RAG pattern to your own documents.

In [ ]:
# Exercise 1: RAG chain for your domain
# TODO:
# 1. Write 3 text files on a topic you choose (cooking, sports, history, tech, etc.)
# 2. Load them with TextLoader
# 3. Split, embed, and create a new Chroma vector store
# 4. Build an LCEL RAG chain with a domain-appropriate system prompt
# 5. Ask 5 questions and evaluate the answers

MY_DOCS_DIR = pathlib.Path("/tmp/my_lc_docs")
MY_DOCS_DIR.mkdir(exist_ok=True)

# TODO: Write your documents and build the chain

---

## 🏋️ Exercise 2: Custom Prompt Template

Modify the RAG prompt to add specific output formatting requirements.

In [ ]:
# Exercise 2: Custom prompt for structured output
# TODO: Modify the RAG_PROMPT template so the answer always returns:
# {
#   "answer": "...",
#   "confidence": "high" | "medium" | "low",
#   "sources": ["filename1.txt", ...]
# }
# Hint: Combine ChatPromptTemplate with response_mime_type in the LLM config
# Or use a structured output schema

STRUCTURED_RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a climate science assistant. Answer using the provided context.
# TODO: Add instructions to return structured JSON output"""),
    ("human", """Context:\n{context}\n\nQuestion: {question}""")
])

# TODO: Build and test the structured chain

---

## LangChain RAG Quick Reference

```python
# The minimal LangChain RAG pattern:

embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages([...])
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

answer = chain.invoke("Your question here")
```

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| LangChain RAG Tutorial | Docs | [python.langchain.com/docs/use_cases/question_answering](https://python.langchain.com/docs/use_cases/question_answering/) |
| LCEL Introduction | Docs | [python.langchain.com/docs/expression_language](https://python.langchain.com/docs/expression_language/) |
| LangChain + Gemini | Docs | [python.langchain.com/docs/integrations/llms/google_ai](https://python.langchain.com/docs/integrations/llms/google_ai/) |

---

**Next up:** [04_RAG_vs_Vanilla_LLM.ipynb](./04_RAG_vs_Vanilla_LLM.ipynb)